# M6 W6 v2 retrieval baseline and candidate handoff

This notebook is the GitHub-ready M6 retrieval deliverable for the current v2 development handoff. It uses exactly three teammate-supplied inputs: the M4 three-textbook structure-aware corpus ZIP, the integrated 188-question Gold JSONL, and the M5 candidate-index ZIP. It does not modify Gold, rebuild M4 chunks, add M7 personalisation reranking, or calculate M8 answer-quality metrics.

The supplied corpus declares `mode=development` and `reportable=false`; the Gold contains 162 `pending` rows and 26 `proposed_test` rows, all answerable. For this M6 run, the user designates the 162 `pending` rows as **Dev** and the remaining 26 `proposed_test` rows as **Test**. The original source labels and `annotation_status` values remain in provenance. These are valid working Dev/Test results for the supplied three-textbook handoff, but they must not be described as the final six-textbook 60/180 evaluation required by the W5-W6 work plan.

Nine dense models are registered: MiniLM, MPNet, E5-base-v2, BGE-base-en-v1.5, Jina-v2-small, GTE-ModernBERT, Nomic-v1.5, BGE-M3, and Qwen3-Embedding-0.6B. The three M5 indexes (GTE-ModernBERT, BGE-M3, Qwen3) are reused only after strict identity and hash checks. The remaining six indexes are built from the exact same 5,976 chunks. BM25 keeps stopword removal, `k1=1.5`, `b=0.75`, query term frequency, and the earlier m6local weighted-RRF tie-break. Each dense model is compared as Dense alone and at Dense/BM25 weights 0.05/0.95, 0.10/0.90, 0.20/0.80, 0.25/0.75, 0.50/0.50, and 0.75/0.25.

Run top-to-bottom. By default the notebook performs all input and provenance checks but does not start the expensive model run. Set `CS30_M6_RUN_MODELS=1` to run all models. Inputs are resolved from `CS30_M6_*` environment variables, `eval_inputs/m6_v2/`, or the current user's Desktop. Outputs go to the ignored `artifacts/m6_w6_retrieval/` directory.

Install with `python -m pip install -e ".[ml,notebook]" pandas`. A CUDA-compatible PyTorch build is strongly recommended; set `CS30_M6_DEVICE=cpu` only if a long CPU run is acceptable.


## Results overview from the completed diagnostic run

The same code and three source artifacts were previously run across all nine embedding models. Each dense model used one Dense row and six Dense/BM25 hybrid rows; BM25 is a separate baseline. The table shows the best development configuration for each model. Higher complete evidence@5 and MRR are better.

| Model | Best mode | Complete evidence@5 | MRR |
|---|---|---:|---:|
| bm25 | bm25 | 133/162 | 0.568 |
| minilm | hybrid_d0.05_b0.95 | 130/162 | 0.552 |
| mpnet | hybrid_d0.05_b0.95 | 129/162 | 0.545 |
| e5_base | hybrid_d0.10_b0.90 | 133/162 | 0.560 |
| bge_base | hybrid_d0.20_b0.80 | 133/162 | 0.546 |
| jina | hybrid_d0.05_b0.95 | 132/162 | 0.562 |
| gte_modernbert | hybrid_d0.05_b0.95 | 135/162 | 0.554 |
| nomic | hybrid_d0.05_b0.95 | 134/162 | 0.569 |
| bge_m3 | hybrid_d0.10_b0.90 | 135/162 | 0.564 |
| qwen3 | hybrid_d0.05_b0.95 | 135/162 | 0.557 |

The Dev-selected dense/hybrid configuration is `bge_m3` / `hybrid_d0.10_b0.90`; it retrieves complete evidence within top 5 for 135/162 Dev questions. Its separately evaluated 26-question Test result is 16/26. This is the working Test result for the supplied three-textbook data, not a formal six-textbook result.


In [ ]:
from __future__ import annotations
import collections, hashlib, json, math, os, re, subprocess, zipfile
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "cs30").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from the cs30-personalised-rag checkout.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())

def resolve_input(env_name: str, filename: str) -> Path:
    candidates = []
    if os.getenv(env_name):
        candidates.append(Path(os.environ[env_name]).expanduser())
    candidates.extend([
        PROJECT_ROOT / "eval_inputs" / "m6_v2" / filename,
        Path.home() / "Desktop" / filename,
    ])
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    raise FileNotFoundError(f"Missing {filename}. Set {env_name} or place it in eval_inputs/m6_v2/.")

CORPUS_ZIP = resolve_input("CS30_M6_CORPUS_ZIP", "v2_structure_aware_corpus_3book_dev_main_3553825.zip")
M5_ZIP = resolve_input("CS30_M6_INDEX_ZIP", "M5_v2_candidate_indexes_2026-10-09.zip")
INTEGRATED_GOLD = resolve_input("CS30_M6_GOLD", "gold_v0_1_1_integrated.jsonl")
OUTPUT_DIR = Path(os.getenv("CS30_M6_OUTPUT_DIR", str(PROJECT_ROOT / "artifacts" / "m6_w6_retrieval"))).expanduser().resolve()
INDEX_DIR = OUTPUT_DIR / "indexes"
MODEL_CACHE = Path(os.getenv("HF_HOME", str(Path.home() / ".cache" / "huggingface"))).expanduser().resolve()

RUN_EXPENSIVE = os.getenv("CS30_M6_RUN_MODELS", "0") == "1"
FORMAL_MODE = os.getenv("CS30_M6_FORMAL_MODE", "0") == "1"
GOLD_ACCEPTANCE_POLICY = "user_designated_dev162_test26"
FROZEN_EVALUATION_SET_VERSION = os.getenv("CS30_M6_EVALUATION_SET_VERSION") or None
FROZEN_GOLD_ANNOTATION_VERSION = os.getenv("CS30_M6_GOLD_ANNOTATION_VERSION") or None
DEVICE = os.getenv("CS30_M6_DEVICE", "cuda")
TOP_K = (1, 3, 5, 10)
SEARCH_DEPTH = 50
RETRIEVAL_DEPTH = max(TOP_K)
RRF_K = 60
HYBRID_WEIGHTS = ((0.95, 0.05), (0.90, 0.10), (0.80, 0.20),
                  (0.75, 0.25), (0.50, 0.50), (0.25, 0.75))  # (BM25, Dense)

try:
    repo_commit = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
    ).strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    repo_commit = "unavailable"

print("Repository:", PROJECT_ROOT)
print("Repository commit:", repo_commit)
print("Corpus ZIP:", CORPUS_ZIP)
print("Index ZIP:", M5_ZIP)
print("Gold:", INTEGRATED_GOLD)
print("Output:", OUTPUT_DIR)
print("Expensive run enabled:", RUN_EXPENSIVE, "device:", DEVICE)


## 1. Model registry and input provenance

In [ ]:
MODEL_SPECS = {
    "minilm": dict(model="sentence-transformers/all-MiniLM-L6-v2", dimension=384, batch=64),
    "mpnet": dict(model="sentence-transformers/all-mpnet-base-v2", dimension=768, batch=32),
    "e5_base": dict(model="intfloat/e5-base-v2", dimension=768, batch=32, query_prefix="query: ", document_prefix="passage: "),
    "bge_base": dict(model="BAAI/bge-base-en-v1.5", dimension=768, batch=32, query_prefix="Represent this sentence for searching relevant passages: "),
    "jina": dict(model="jinaai/jina-embeddings-v2-small-en", dimension=512, batch=16, trust_remote_code=True),
    "gte_modernbert": dict(model="Alibaba-NLP/gte-modernbert-base", dimension=768, batch=4, trust_remote_code=True, m5_path="gte-modernbert"),
    "nomic": dict(model="nomic-ai/nomic-embed-text-v1.5", dimension=768, batch=8, trust_remote_code=True, query_prefix="search_query: ", document_prefix="search_document: "),
    "bge_m3": dict(model="BAAI/bge-m3", dimension=1024, batch=4, m5_path="bge-m3"),
    "qwen3": dict(model="Qwen/Qwen3-Embedding-0.6B", dimension=1024, batch=2, m5_path="qwen3-embedding"),
}
for spec in MODEL_SPECS.values():
    spec.setdefault("query_prefix", "")
    spec.setdefault("document_prefix", "")
    spec.setdefault("trust_remote_code", False)
assert len(MODEL_SPECS) == 9 and len({spec["model"] for spec in MODEL_SPECS.values()}) == 9

def sha256(data: bytes) -> str:
    return "sha256:" + hashlib.sha256(data).hexdigest()

def read_jsonl_bytes(data: bytes) -> list[dict]:
    return [json.loads(line) for line in data.decode("utf-8-sig").splitlines() if line.strip()]

with zipfile.ZipFile(CORPUS_ZIP) as archive:
    corpus_bytes = archive.read("records.jsonl")
    manifest_bytes = archive.read("manifest.json")
    corpus_manifest = json.loads(manifest_bytes)
    chunks = read_jsonl_bytes(corpus_bytes)
    delivered_checksums = archive.read("SHA256SUMS.txt").decode("utf-8")
    for line in delivered_checksums.splitlines():
        expected, filename = line.split(None, 1)
        assert hashlib.sha256(archive.read(filename.strip())).hexdigest() == expected, filename

assert sha256(corpus_bytes) == corpus_manifest["corpus_hash"]
assert len(chunks) == corpus_manifest["record_count"] == 5976
assert len({chunk["chunk_id"] for chunk in chunks}) == len(chunks)
assert {chunk["textbook_id"] for chunk in chunks} == set(corpus_manifest["included_textbook_ids"])
assert corpus_manifest["reportable"] is False and corpus_manifest["mode"] == "development"

gold_bytes = INTEGRATED_GOLD.read_bytes()
gold = read_jsonl_bytes(gold_bytes)
assert len(gold) == 188 and len({row["question_id"] for row in gold}) == len(gold)
assert {row["corpus_version"] for row in gold} == {corpus_manifest["corpus_version"]}
assert GOLD_ACCEPTANCE_POLICY == "user_designated_dev162_test26"
assert all(row["answerable"] is True for row in gold)

print({
    "corpus_version": corpus_manifest["corpus_version"],
    "chunks": len(chunks),
    "textbooks": corpus_manifest["included_textbook_ids"],
    "integrated_gold_rows": len(gold),
    "integrated_gold_sha256": sha256(gold_bytes),
    "gold_acceptance_policy": GOLD_ACCEPTANCE_POLICY,
    "source_annotation_statuses": dict(collections.Counter(row["annotation_status"] for row in gold)),
    "gold_splits": dict(collections.Counter(row["split"] for row in gold)),
    "registered_dense_models": [spec["model"] for spec in MODEL_SPECS.values()],
})


## 2. Bind evidence by source `block_id`

M3's chapter-local offsets and M4's document-wide offsets are different coordinate systems. A Gold span is accepted only when its block ID resolves uniquely, its chapter matches, and its normalized verbatim text occurs in that chunk. Fail closed if any span cannot be bound.

In [ ]:
def normalized(text: str) -> str:
    return " ".join(text.casefold().split())

block_to_chunks = collections.defaultdict(set)
by_chunk = {c["chunk_id"]: c for c in chunks}
for chunk in chunks:
    for span in chunk["spans"]:
        block_to_chunks[span["block_id"]].add(chunk["chunk_id"])

binding_rows = []
gold_paths = {}
for question in gold:
    qid = question["question_id"]
    paths = []
    for group in question["gold_core_evidence_sets"]:
        mapped = []
        for span in group:
            candidates = block_to_chunks.get(span.get("block_id"), set())
            assert len(candidates) == 1, (qid, span.get("span_id"), "non-unique/missing block")
            chunk_id = next(iter(candidates))
            chunk = by_chunk[chunk_id]
            assert str(chunk["chapter_id"]) == str(span["chapter_id"]), (qid, "chapter mismatch")
            assert normalized(span["verbatim_text"]) in normalized(chunk["text"]), (qid, "verbatim mismatch")
            mapped.append(chunk_id)
            binding_rows.append({"question_id": qid, "span_id": span["span_id"],
                                 "block_id": span["block_id"], "chunk_id": chunk_id,
                                 "textbook_id": chunk["textbook_id"], "method": "unique_block_id+chapter+verbatim"})
        paths.append(frozenset(mapped))
    assert paths, (qid, "no core evidence")
    gold_paths[qid] = paths
binding_bytes = ("\n".join(json.dumps(r, sort_keys=True, ensure_ascii=False) for r in binding_rows) + "\n").encode("utf-8")
print("Bound spans:", len(binding_rows), "of", sum(len(p) for q in gold for p in q["gold_core_evidence_sets"]))
print("Binding SHA-256:", sha256(binding_bytes))

## 3. Verify the three M5 candidate index artifacts

In [ ]:
M5_ARTIFACTS = {}
M5_INDEX_BYTES = {}
ordered_chunk_ids = [c["chunk_id"] for c in chunks]
with zipfile.ZipFile(M5_ZIP) as z:
    # ZIP creators may store member names with either / or \\ separators.
    # Index by a normalized POSIX-style name, but read the exact stored name.
    m5_member_names = {}
    for stored_name in z.namelist():
        normalized_name = stored_name.replace("\\", "/")
        assert normalized_name not in m5_member_names, f"Ambiguous ZIP member: {normalized_name}"
        m5_member_names[normalized_name] = stored_name

    def read_m5_member(normalized_name: str) -> bytes:
        if normalized_name not in m5_member_names:
            raise KeyError(f"Missing M5 ZIP member: {normalized_name}")
        return z.read(m5_member_names[normalized_name])

    for key, spec in MODEL_SPECS.items():
        if "m5_path" not in spec:
            continue
        prefix = f"v2-m5-candidates/{spec['m5_path']}/"
        artifact = json.loads(read_m5_member(prefix + "artifact.json"))
        ids_bytes = read_m5_member(prefix + "index/chunk_ids.json")
        idx_bytes = read_m5_member(prefix + "index/index.faiss")
        for name in ("corpus_version", "corpus_hash", "manifest_hash", "chunk_config_hash"):
            assert artifact[name] == corpus_manifest[name], (key, name)
        assert artifact["required_textbook_ids"] == corpus_manifest["required_textbook_ids"]
        assert artifact["included_textbook_ids"] == corpus_manifest["included_textbook_ids"]
        assert artifact["chunk_count"] == len(chunks)
        assert artifact["chunk_ids"] == ordered_chunk_ids
        assert json.loads(ids_bytes) == ordered_chunk_ids
        assert sha256(ids_bytes) == artifact["metadata"]["chunk_ids_sha256"]
        assert sha256(idx_bytes) == artifact["metadata"]["index_sha256"]
        assert artifact["embedding_model"] == spec["model"]
        assert artifact["embedding_dimension"] == spec["dimension"]
        assert artifact["similarity_metric"] == "inner_product"
        assert artifact["normalise_embeddings"] is True
        assert artifact["index_type"] == "faiss.IndexFlatIP"
        assert not spec["query_prefix"] and not spec["document_prefix"]  # M5 encoder uses plain encode.
        M5_ARTIFACTS[key] = artifact
        M5_INDEX_BYTES[key] = idx_bytes
print(pd.DataFrame([{"model": a["embedding_model"], "revision": a["embedding_revision"],
                     "dimension": a["embedding_dimension"], "chunks": a["chunk_count"]}
                    for a in M5_ARTIFACTS.values()]).to_string(index=False))

## 4. Strict formal gate and diagnostic split

In [ ]:
EVALUATION_SET_VERSION = "m6-v2-working-dev162-test26-2026-10-12"
GOLD_ANNOTATION_VERSION = sorted({row["gold_annotation_version"] for row in gold})
WORKING_SPLIT_POLICY = {"pending": "dev", "proposed_test": "test", "dev": "dev", "test": "test"}

def assert_formal_ready() -> None:
    assert corpus_manifest["reportable"] is True, "M4 corpus is development-only."
    assert corpus_manifest["mode"] != "development"
    assert len(corpus_manifest["included_textbook_ids"]) == 6, "Formal W6 requires six textbooks."
    assert len(gold) == 240, "Formal W6 requires 240 Gold questions."
    assert collections.Counter(row["split"] for row in gold) == {"dev": 60, "test": 180}
    assert all(row["corpus_version"] == corpus_manifest["corpus_version"] for row in gold)
    assert all(row["annotation_status"] == "reviewed" for row in gold)
    assert FROZEN_EVALUATION_SET_VERSION, "Missing approved evaluation_set_version."
    assert FROZEN_GOLD_ANNOTATION_VERSION, "Missing frozen Gold annotation version."
    assert GOLD_ANNOTATION_VERSION == [FROZEN_GOLD_ANNOTATION_VERSION]

if FORMAL_MODE:
    assert_formal_ready()
else:
    print("Gold accepted for M6 scoring; original annotation statuses remain in provenance.")
    try:
        assert_formal_ready()
    except AssertionError as exc:
        print("Formal gate correctly rejects the current three-textbook development inputs:", exc)
    else:
        raise AssertionError("Formal gate unexpectedly accepted development inputs")

dev_gold = [row for row in gold if WORKING_SPLIT_POLICY.get(row["split"]) == "dev"]
test_gold = [row for row in gold if WORKING_SPLIT_POLICY.get(row["split"]) == "test"]
assert len(dev_gold) == 162 and len(test_gold) == 26
assert not ({row["question_id"] for row in dev_gold} & {row["question_id"] for row in test_gold})
print("Working Dev:", len(dev_gold), "Working Test:", len(test_gold))


## 5. BM25, dense and weighted RRF (m6local retrieval method)

In [ ]:
STOPWORDS = frozenset({
    "a", "an", "and", "are", "as", "at", "be", "by", "can", "do", "does",
    "for", "from", "how", "in", "is", "it", "of", "on", "or", "that", "the",
    "to", "was", "were", "what", "when", "where", "which", "who", "why",
    "will", "with",
})
REMOVE_BM25_STOPWORDS = True
BM25_K1 = 1.5
BM25_B = 0.75
TOKEN_RE = re.compile(r"[a-z0-9]+(?:['-][a-z0-9]+)*", re.IGNORECASE)
def tokenize(text: str, remove_stopwords: bool = REMOVE_BM25_STOPWORDS) -> list[str]:
    tokens = TOKEN_RE.findall(text.casefold())
    return [token for token in tokens if token not in STOPWORDS] if remove_stopwords else tokens

doc_tokens = [tokenize(c["text"]) for c in chunks]
doc_lengths = np.asarray([len(t) for t in doc_tokens], dtype=np.float64)
avg_doc_length = float(doc_lengths.mean())
postings = collections.defaultdict(list)
for i, tokens in enumerate(doc_tokens):
    for term, freq in collections.Counter(tokens).items():
        postings[term].append((i, freq))

def bm25_search(query: str, depth: int = SEARCH_DEPTH) -> list[tuple[str, float]]:
    scores = collections.defaultdict(float)
    for term, query_freq in collections.Counter(tokenize(query)).items():
        rows = postings.get(term, ())
        df = len(rows)
        if not df:
            continue
        idf = math.log(1.0 + (len(chunks) - df + 0.5) / (df + 0.5))
        for row, freq in rows:
            denom = freq + BM25_K1 * (1.0 - BM25_B + BM25_B * doc_lengths[row] / avg_doc_length)
            score = query_freq * idf * freq * (BM25_K1 + 1.0) / denom
            scores[row] += score
    ranked = sorted(scores, key=lambda i: (-scores[i], ordered_chunk_ids[i]))[:depth]
    return [(ordered_chunk_ids[i], float(scores[i])) for i in ranked]

def weighted_rrf(bm25_hits, dense_hits, bm25_weight: float, dense_weight: float,
                 depth: int = RETRIEVAL_DEPTH) -> list[tuple[str, float]]:
    assert math.isclose(bm25_weight + dense_weight, 1.0, abs_tol=1e-9)
    fused = collections.defaultdict(float)
    best_rank = {}
    for weight, hits in ((dense_weight, dense_hits), (bm25_weight, bm25_hits)):
        for rank, (chunk_id, _) in enumerate(hits[:SEARCH_DEPTH], 1):
            fused[chunk_id] += weight / (RRF_K + rank)
            best_rank[chunk_id] = min(rank, best_rank.get(chunk_id, math.inf))
    return sorted(fused.items(), key=lambda row: (-row[1], best_rank[row[0]], row[0]))[:depth]

assert tokenize("The force and force") == ["force", "force"]
assert tokenize("the and") == []
METHOD_CONFIG = {
    "bm25_stopwords_enabled": REMOVE_BM25_STOPWORDS,
    "bm25_stopwords": sorted(STOPWORDS),
    "bm25_k1": BM25_K1,
    "bm25_b": BM25_B,
    "bm25_query_term_frequency": True,
    "rrf_k": RRF_K,
    "rrf_tie_break": "minimum_candidate_rank_then_chunk_id",
    "candidate_depth": SEARCH_DEPTH,
    "final_depth": RETRIEVAL_DEPTH,
}
print("BM25 stopwords:", REMOVE_BM25_STOPWORDS, "k1:", BM25_K1, "b:", BM25_B)

def load_or_build_dense(model_key: str):
    # Return (SentenceTransformer, FAISS index, artifact); write only compatible local indexes.
    import faiss
    from sentence_transformers import SentenceTransformer
    spec = MODEL_SPECS[model_key]
    revision = M5_ARTIFACTS[model_key]["embedding_revision"] if model_key in M5_ARTIFACTS else None
    model = SentenceTransformer(spec["model"], revision=revision, device=DEVICE,
                                trust_remote_code=spec["trust_remote_code"], cache_folder=str(MODEL_CACHE))
    if model_key in M5_ARTIFACTS:
        artifact = M5_ARTIFACTS[model_key]
        index = faiss.deserialize_index(np.frombuffer(M5_INDEX_BYTES[model_key], dtype="uint8"))
    else:
        location = INDEX_DIR / model_key
        artifact_path = location / "artifact.json"
        index_path = location / "index.faiss"
        policy = {"corpus_hash": corpus_manifest["corpus_hash"], "model": spec["model"],
                  "query_prefix": spec["query_prefix"], "document_prefix": spec["document_prefix"],
                  "metric": "inner_product", "normalise_embeddings": True,
                  "chunk_ids_sha256": sha256(json.dumps(ordered_chunk_ids, separators=(',', ':')).encode("utf-8"))}
        if artifact_path.is_file() and index_path.is_file():
            artifact = json.loads(artifact_path.read_text(encoding="utf-8"))
            assert all(artifact.get(k) == v for k, v in policy.items()), f"Stale local index: {location}"
            assert sha256(index_path.read_bytes()) == artifact["index_sha256"]
            index = faiss.read_index(str(index_path))
        else:
            texts = [spec["document_prefix"] + (c.get("embed_text") or c["text"]) for c in chunks]
            vectors = model.encode(texts, batch_size=spec["batch"], normalize_embeddings=True,
                                   convert_to_numpy=True, show_progress_bar=True).astype("float32")
            assert vectors.shape == (len(chunks), spec["dimension"])
            assert np.isfinite(vectors).all()
            faiss.normalize_L2(vectors)
            index = faiss.IndexFlatIP(spec["dimension"])
            index.add(vectors)
            location.mkdir(parents=True, exist_ok=True)
            faiss.write_index(index, str(index_path))
            resolved = getattr(getattr(getattr(model[0], "auto_model", None), "config", None), "_commit_hash", None)
            artifact = {**policy, "embedding_revision": str(resolved or "unresolved"),
                        "dimension": spec["dimension"], "chunk_count": len(chunks),
                        "index_sha256": sha256(index_path.read_bytes()), "built_at_utc": datetime.now(timezone.utc).isoformat()}
            artifact_path.write_text(json.dumps(artifact, indent=2), encoding="utf-8")
    assert index.ntotal == len(chunks) and index.d == spec["dimension"]
    assert isinstance(index, faiss.IndexFlatIP), "Only normalized FlatIP is supported."
    return model, index, artifact

def dense_search(query: str, model_key: str, model, index, depth: int = SEARCH_DEPTH):
    import faiss
    spec = MODEL_SPECS[model_key]
    query_vec = model.encode([spec["query_prefix"] + query], normalize_embeddings=True,
                             convert_to_numpy=True, show_progress_bar=False).astype("float32")
    assert query_vec.shape == (1, spec["dimension"]) and np.isfinite(query_vec).all()
    faiss.normalize_L2(query_vec)  # Matches artifact normalise_embeddings=True and FlatIP.
    scores, positions = index.search(query_vec, min(depth, len(chunks)))
    return [(ordered_chunk_ids[int(i)], float(score)) for score, i in zip(scores[0], positions[0]) if i >= 0]

## 6. Evidence retrieval metrics and grouping

In [ ]:
def score_one(question: dict, hits: list[tuple[str, float]], model_key: str,
              mode: str, bm25_weight=None, dense_weight=None) -> dict:
    ids = [chunk_id for chunk_id, _ in hits]
    assert len(ids) == len(set(ids)), (question["question_id"], model_key, mode, "duplicate hits")
    qid = question["question_id"]
    answerable = question.get("answerable", True) is True
    paths = gold_paths[qid] if answerable else []
    relevant = set().union(*paths) if paths else set()
    rank = {chunk_id: i for i, chunk_id in enumerate(ids, 1)}
    first_rank = min((rank[cid] for cid in relevant if cid in rank), default=None)
    complete_rank = min((max((rank.get(cid, math.inf) for cid in path), default=math.inf)
                         for path in paths), default=math.inf)
    complete_rank = None if math.isinf(complete_rank) else int(complete_rank)
    source_ids = {by_chunk[cid]["textbook_id"] for cid in relevant}
    providers = {by_chunk[cid]["provider"] for cid in relevant}
    row = {"question_id": qid, "source_split": question["source_split"],
           "source_evaluation_split": question["split"], "evaluation_split": WORKING_SPLIT_POLICY[question["split"]], "annotation_status": question["annotation_status"],
           "model": model_key, "mode": mode, "bm25_weight": bm25_weight,
           "dense_weight": dense_weight, "answerable": answerable,
           "scope": ("unanswerable" if not answerable else "cross_textbook" if len(source_ids) > 1 else "single_source"),
           "provider_scope": ("unanswerable" if not answerable else "cross_provider" if len(providers) > 1 else "single_provider"),
           "gold_textbook": "+".join(sorted(source_ids)), "gold_provider": "+".join(sorted(providers)),
           "first_relevant_rank": first_rank, "complete_evidence_rank": complete_rank,
           "mrr": 0.0 if first_rank is None else 1 / first_rank,
           "complete_evidence_mrr": 0.0 if complete_rank is None else 1 / complete_rank,
           "returned_count": len(hits), "empty_retrieval": int(not hits),
           "unanswerable_no_hit": int(not hits) if not answerable else None,
           "unanswerable_false_positive": int(bool(hits)) if not answerable else None,
           "ranked_chunk_ids": ids}
    for k in TOP_K:
        top = set(ids[:k])
        row[f"hit_at_{k}"] = int(bool(top & relevant))
        row[f"complete_hit_at_{k}"] = int(any(path <= top for path in paths))
        row[f"evidence_recall_at_{k}"] = max((len(path & top) / len(path) for path in paths), default=0.0) if answerable else None
        row[f"core_precision_at_{k}"] = len(top & relevant) / len(top) if top and answerable else None
        row[f"no_relevant_hit_at_{k}"] = int(not (top & relevant)) if answerable else None
    return row

def summarize(rows: list[dict]) -> pd.DataFrame:
    frame = pd.DataFrame(rows)
    # The integrated Gold has no partial-evidence spans. Thus noise is exactly
    # 1 - core precision for nonempty retrieval, and partial count is zero.
    assert all(not q.get("partial_evidence") for q in gold)
    frame["run_name"] = frame["model"] + "_" + frame["mode"]
    frame["first_hit_mrr"] = frame["mrr"]
    for k in TOP_K:
        frame[f"complete_evidence_hit_at_{k}"] = frame[f"complete_hit_at_{k}"]
        frame[f"recall_at_{k}"] = frame[f"evidence_recall_at_{k}"]
        frame[f"returned_count_at_{k}"] = frame["returned_count"].clip(upper=k)
        frame[f"empty_retrieval_rate_at_{k}"] = frame[f"returned_count_at_{k}"].eq(0).astype(float)
        frame[f"no_hit_rate_at_{k}"] = frame[f"empty_retrieval_rate_at_{k}"]
        frame[f"no_relevant_hit_rate_at_{k}"] = frame[f"no_relevant_hit_at_{k}"].where(
            frame[f"returned_count_at_{k}"].gt(0))
        frame[f"context_noise_rate_at_{k}"] = 1 - frame[f"core_precision_at_{k}"]
        frame[f"partial_evidence_count_at_{k}"] = 0
    metrics = ["mrr", "complete_evidence_mrr"] + [
        f"{metric}_at_{k}" for metric in ("hit", "complete_evidence_hit", "evidence_recall") for k in TOP_K
    ] + ["first_hit_mrr"] + [
        f"{metric}_at_{k}" for metric in ("complete_hit", "recall", "core_precision",
            "context_noise_rate", "no_relevant_hit_rate", "partial_evidence_count",
            "returned_count", "empty_retrieval_rate", "no_hit_rate") for k in TOP_K
    ] + ["returned_count", "empty_retrieval", "unanswerable_no_hit", "unanswerable_false_positive"] + [
        f"no_relevant_hit_at_{k}" for k in TOP_K]
    groups = ["run_name", "model", "mode", "dense_weight", "bm25_weight"]
    grouped = frame.groupby(groups, dropna=False)
    counts = grouped.size().rename("question_count")
    nullable = [f"{metric}_at_{k}" for metric in ("core_precision", "context_noise_rate",
        "no_relevant_hit_rate", "partial_evidence_count") for k in TOP_K]
    defined = grouped[nullable].count().rename(columns={column: f"{column}_defined_count" for column in nullable})
    return counts.to_frame().join(grouped[metrics].mean()).join(defined).reset_index()

def grouped_summary(rows: list[dict]) -> pd.DataFrame:
    frame = pd.DataFrame(rows)
    groups = ["model", "mode", "evaluation_split", "scope", "provider_scope", "gold_textbook", "gold_provider"]
    return frame.groupby(groups, dropna=False).agg(question_count=("question_id", "nunique"),
        hit_at_5=("hit_at_5", "mean"), complete_hit_at_5=("complete_hit_at_5", "mean"),
        evidence_recall_at_5=("evidence_recall_at_5", "mean"), mrr=("mrr", "mean")).reset_index()

# Tiny synthetic tests only verify metric logic; they are never merged with real output.
_q = {"question_id": dev_gold[0]["question_id"], "source_split": "train", "split": "pending", "annotation_status": dev_gold[0]["annotation_status"]}
_a = next(iter(gold_paths[_q["question_id"]][0]))
assert score_one(_q, [(_a, 1.0)], "fixture", "fixture")["hit_at_1"] == 1
assert score_one(_q, [], "fixture", "fixture")["empty_retrieval"] == 1
assert score_one(_q, [], "fixture", "fixture")["no_relevant_hit_at_5"] == 1
_u = {**_q, "answerable": False}
assert score_one(_u, [], "fixture", "fixture")["unanswerable_no_hit"] == 1
assert score_one(_u, [(_a, 1.0)], "fixture", "fixture")["unanswerable_false_positive"] == 1
_joint = next(q for q in gold if any(len(path) > 1 for path in gold_paths[q["question_id"]]))
_joint_ids = list(next(path for path in gold_paths[_joint["question_id"]] if len(path) > 1))
assert score_one(_joint, [(_joint_ids[0], 1.0)], "fixture", "fixture")["hit_at_1"] == 1
assert score_one(_joint, [(_joint_ids[0], 1.0)], "fixture", "fixture")["complete_hit_at_1"] == 0
assert score_one(_joint, [(cid, 1.0) for cid in _joint_ids], "fixture", "fixture")["complete_hit_at_3"] == 1
print("Metric smoke tests passed.")

## 7. Run real diagnostic retrieval

`RUN_EXPENSIVE=False` performs all input, mapping and index-provenance checks above but **does not** evaluate models. When enabled, every model is run on the 162 user-designated working Dev rows. The 26 source rows labelled `proposed_test` are the working Test set and are not used for configuration selection. No unanswerable, cross-textbook or cross-provider rate is claimed when the dataset lacks those cases.

In [ ]:
def evaluate_questions(questions: list[dict], include_models: list[str], checkpoint_prefix: str = "dev"):
    rows, candidates, index_provenance = [], [], {}
    assert checkpoint_prefix in ("dev", "test_all"), checkpoint_prefix
    progress_path = OUTPUT_DIR / ("progress.json" if checkpoint_prefix == "dev" else f"{checkpoint_prefix}_progress.json")
    previous = json.loads(progress_path.read_text(encoding="utf-8")) if progress_path.is_file() else {}
    if previous:
        saved_prefix = previous.get("checkpoint_prefix", checkpoint_prefix)
        assert saved_prefix == checkpoint_prefix or (checkpoint_prefix == "test_all" and saved_prefix == "proposed_test_all")
        assert previous["corpus_hash"] == corpus_manifest["corpus_hash"]
        assert previous["gold_sha256"] == sha256(gold_bytes)
        assert previous["total_dense_models"] == len(include_models)
        assert previous["hybrid_weights"] == [list(pair) for pair in HYBRID_WEIGHTS]
        assert previous["method_config"] == METHOD_CONFIG
    completed_models = list(previous.get("completed_models", []))
    def recover(key: str):
        if key not in completed_models:
            return None
        metrics_path = OUTPUT_DIR / f"{checkpoint_prefix}_{key}_metrics.jsonl"
        candidates_path = OUTPUT_DIR / f"{checkpoint_prefix}_{key}_candidate_hits.jsonl"
        assert metrics_path.is_file() and candidates_path.is_file(), key
        old_rows = read_jsonl_bytes(metrics_path.read_bytes())
        old_candidates = read_jsonl_bytes(candidates_path.read_bytes())
        for row in old_rows:
            source_label = row.get("source_evaluation_split", row.get("evaluation_split"))
            row["source_evaluation_split"] = source_label
            row["evaluation_split"] = WORKING_SPLIT_POLICY[source_label]
        expected = len(questions) if key == "bm25" else len(questions) * (1 + len(HYBRID_WEIGHTS))
        assert len(old_rows) == expected and {r["question_id"] for r in old_rows} == {q["question_id"] for q in questions}
        return old_rows, old_candidates
    def checkpoint(key: str, new_rows: list[dict], new_candidates: list[dict]):
        # Keep a per-model recovery point; no result is marked complete before its model finishes.
        (OUTPUT_DIR / f"{checkpoint_prefix}_{key}_metrics.jsonl").write_text(
            "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in new_rows), encoding="utf-8")
        (OUTPUT_DIR / f"{checkpoint_prefix}_{key}_candidate_hits.jsonl").write_text(
            "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in new_candidates), encoding="utf-8")
        if key not in completed_models:
            completed_models.append(key)
        progress_path.write_text(json.dumps({
            "completed_models": completed_models, "total_dense_models": len(include_models),
            "corpus_hash": corpus_manifest["corpus_hash"], "gold_sha256": sha256(gold_bytes),
            "hybrid_weights": [list(pair) for pair in HYBRID_WEIGHTS],
            "method_config": METHOD_CONFIG,
            "checkpoint_prefix": checkpoint_prefix,
            "updated_at_utc": datetime.now(timezone.utc).isoformat()}, indent=2), encoding="utf-8")
    bm25_by_qid = {q["question_id"]: bm25_search(q["question"], SEARCH_DEPTH) for q in questions}
    recovered = recover("bm25")
    if recovered:
        rows.extend(recovered[0]); candidates.extend(recovered[1])
        print("Recovered completed bm25 checkpoint", flush=True)
    else:
        for q in questions:
            hits = bm25_by_qid[q["question_id"]][:RETRIEVAL_DEPTH]
            rows.append(score_one(q, hits, "bm25", "bm25"))
            for rank, (chunk_id, score) in enumerate(hits, 1):
                candidates.append({"question_id": q["question_id"], "model": "bm25", "mode": "bm25",
                                   "rank": rank, "chunk_id": chunk_id, "score": score,
                                   "query": q["question"], "corpus_hash": corpus_manifest["corpus_hash"],
                                   "textbook_id": by_chunk[chunk_id]["textbook_id"],
                                   "provider": by_chunk[chunk_id]["provider"],
                                   "source_locator": by_chunk[chunk_id]["source_locator"]})
        checkpoint("bm25", rows, candidates)
    for key in include_models:
        recovered = recover(key)
        if recovered:
            rows.extend(recovered[0]); candidates.extend(recovered[1])
            index_provenance[key] = (M5_ARTIFACTS[key] if key in M5_ARTIFACTS else
                json.loads((INDEX_DIR / key / "artifact.json").read_text(encoding="utf-8")))
            print("Recovered completed", key, "checkpoint", flush=True)
            continue
        print("Running", key, MODEL_SPECS[key]["model"], flush=True)
        row_start, candidate_start = len(rows), len(candidates)
        model, index, artifact = load_or_build_dense(key)
        index_provenance[key] = artifact
        for q in questions:
            qid = q["question_id"]
            dense = dense_search(q["question"], key, model, index, SEARCH_DEPTH)
            variants = [("dense", None, None, dense[:RETRIEVAL_DEPTH])]
            variants += [(f"hybrid_d{dw:.2f}_b{bw:.2f}", bw, dw, weighted_rrf(bm25_by_qid[qid], dense, bw, dw))
                         for bw, dw in HYBRID_WEIGHTS]
            for mode, bw, dw, hits in variants:
                rows.append(score_one(q, hits, key, mode, bw, dw))
                for rank, (chunk_id, score) in enumerate(hits, 1):
                    candidates.append({"question_id": qid, "model": key, "mode": mode,
                                       "rank": rank, "chunk_id": chunk_id, "score": score,
                                       "query": q["question"], "corpus_hash": corpus_manifest["corpus_hash"],
                                       "textbook_id": by_chunk[chunk_id]["textbook_id"],
                                       "provider": by_chunk[chunk_id]["provider"],
                                       "source_locator": by_chunk[chunk_id]["source_locator"]})
        checkpoint(key, rows[row_start:], candidates[candidate_start:])
        del model, index
        try:
            import torch
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        except ImportError:
            pass
    return rows, candidates, index_provenance

if RUN_EXPENSIVE:
    import faiss, sentence_transformers, torch
    from IPython.display import display
    if DEVICE == "cuda": assert torch.cuda.is_available(), "CUDA requested but unavailable."
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    INDEX_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_CACHE.mkdir(parents=True, exist_ok=True)
    dev_rows, dev_candidates, index_provenance = evaluate_questions(dev_gold, list(MODEL_SPECS))
    dev_summary = summarize(dev_rows).sort_values(["complete_hit_at_5", "mrr"], ascending=False)
    display(dev_summary)
else:
    dev_rows = dev_candidates = []
    index_provenance = {}
    print("Skipped real retrieval. Set RUN_EXPENSIVE=True and rerun this notebook to obtain measured results.")

## 8. Per-model development results (m6local layout)

Each model has one pure Dense row and six hybrid rows. The weight columns always mean **Dense / BM25**, in that order: 0.05/0.95, 0.10/0.90, 0.20/0.80, 0.25/0.75, 0.50/0.50, 0.75/0.25. BM25 alone is shown separately. These are 162-question development results, not final Test results.

The wide tables include every available m6local metric (and v2-specific diagnostics), including the defined-question counts for nullable metrics. The displayed values are cached from `dev_summary.csv`; rerun the cells to recompute them.

In [ ]:
from IPython.display import display

def model_table(key: str, split: str = "dev") -> pd.DataFrame:
    assert split in ("dev", "test_all"), split
    live_name = "dev_summary" if split == "dev" else "test_all_summary"
    if live_name not in globals():
        saved = OUTPUT_DIR / f"{split}_summary.csv"
        if not saved.is_file():
            raise FileNotFoundError(f"Run the full comparison first: {saved}")
        frame = pd.read_csv(saved)
    else:
        frame = globals()[live_name]
    table = frame.loc[frame["model"].eq(key)].copy()
    if table.empty:
        raise ValueError(f"No measured rows for {key}")
    if "run_name" not in table:
        table["run_name"] = table["model"] + "_" + table["mode"]
    table["sort_weight"] = table["dense_weight"].fillna(-1)
    table = table.sort_values("sort_weight")
    table["mode"] = table["mode"].map(lambda mode: "hybrid" if mode.startswith("hybrid_") else mode)
    leading = ["run_name", "model", "mode", "dense_weight", "bm25_weight", "question_count"]
    columns = leading + [column for column in table.columns if column not in leading + ["sort_weight"]]
    return table[columns].reset_index(drop=True)

### Dev: BM25 baseline (162 questions)

In [ ]:
if RUN_EXPENSIVE or (OUTPUT_DIR / "dev_summary.csv").is_file():
    display(model_table("bm25"))
else:
    print("Dev results pending: BM25 baseline")

Selected-metric preview of the saved results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| bm25_bm25 | bm25 | N/A | N/A | 0.568 | 133/162 | 133/162 |

### Dev: MiniLM (162 questions)

In [ ]:
if RUN_EXPENSIVE or (OUTPUT_DIR / "dev_summary.csv").is_file():
    display(model_table("minilm"))
else:
    print("Dev results pending: MiniLM")

Selected-metric preview of the saved results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| minilm_dense | dense | N/A | N/A | 0.294 | 80/162 | 81/162 |
| minilm_hybrid_d0.05_b0.95 | hybrid | 0.05 | 0.95 | 0.552 | 130/162 | 130/162 |
| minilm_hybrid_d0.10_b0.90 | hybrid | 0.10 | 0.90 | 0.520 | 124/162 | 124/162 |
| minilm_hybrid_d0.20_b0.80 | hybrid | 0.20 | 0.80 | 0.496 | 114/162 | 114/162 |
| minilm_hybrid_d0.25_b0.75 | hybrid | 0.25 | 0.75 | 0.485 | 112/162 | 112/162 |
| minilm_hybrid_d0.50_b0.50 | hybrid | 0.50 | 0.50 | 0.432 | 106/162 | 107/162 |
| minilm_hybrid_d0.75_b0.25 | hybrid | 0.75 | 0.25 | 0.352 | 94/162 | 95/162 |

### Dev: MPNet (162 questions)

In [ ]:
if RUN_EXPENSIVE or (OUTPUT_DIR / "dev_summary.csv").is_file():
    display(model_table("mpnet"))
else:
    print("Dev results pending: MPNet")

Selected-metric preview of the saved results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| mpnet_dense | dense | N/A | N/A | 0.301 | 84/162 | 85/162 |
| mpnet_hybrid_d0.05_b0.95 | hybrid | 0.05 | 0.95 | 0.545 | 129/162 | 129/162 |
| mpnet_hybrid_d0.10_b0.90 | hybrid | 0.10 | 0.90 | 0.529 | 126/162 | 126/162 |
| mpnet_hybrid_d0.20_b0.80 | hybrid | 0.20 | 0.80 | 0.512 | 119/162 | 119/162 |
| mpnet_hybrid_d0.25_b0.75 | hybrid | 0.25 | 0.75 | 0.505 | 118/162 | 118/162 |
| mpnet_hybrid_d0.50_b0.50 | hybrid | 0.50 | 0.50 | 0.443 | 106/162 | 107/162 |
| mpnet_hybrid_d0.75_b0.25 | hybrid | 0.75 | 0.25 | 0.356 | 98/162 | 99/162 |

### Dev: E5-base-v2 (162 questions)

In [ ]:
if RUN_EXPENSIVE or (OUTPUT_DIR / "dev_summary.csv").is_file():
    display(model_table("e5_base"))
else:
    print("Dev results pending: E5-base-v2")

Selected-metric preview of the saved results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| e5_base_dense | dense | N/A | N/A | 0.338 | 92/162 | 92/162 |
| e5_base_hybrid_d0.05_b0.95 | hybrid | 0.05 | 0.95 | 0.559 | 133/162 | 133/162 |
| e5_base_hybrid_d0.10_b0.90 | hybrid | 0.10 | 0.90 | 0.560 | 133/162 | 133/162 |
| e5_base_hybrid_d0.20_b0.80 | hybrid | 0.20 | 0.80 | 0.538 | 131/162 | 131/162 |
| e5_base_hybrid_d0.25_b0.75 | hybrid | 0.25 | 0.75 | 0.525 | 130/162 | 130/162 |
| e5_base_hybrid_d0.50_b0.50 | hybrid | 0.50 | 0.50 | 0.479 | 124/162 | 124/162 |
| e5_base_hybrid_d0.75_b0.25 | hybrid | 0.75 | 0.25 | 0.390 | 108/162 | 108/162 |

### Dev: BGE-base-en-v1.5 (162 questions)

In [ ]:
if RUN_EXPENSIVE or (OUTPUT_DIR / "dev_summary.csv").is_file():
    display(model_table("bge_base"))
else:
    print("Dev results pending: BGE-base-en-v1.5")

Selected-metric preview of the saved results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| bge_base_dense | dense | N/A | N/A | 0.376 | 96/162 | 96/162 |
| bge_base_hybrid_d0.05_b0.95 | hybrid | 0.05 | 0.95 | 0.569 | 131/162 | 131/162 |
| bge_base_hybrid_d0.10_b0.90 | hybrid | 0.10 | 0.90 | 0.564 | 132/162 | 132/162 |
| bge_base_hybrid_d0.20_b0.80 | hybrid | 0.20 | 0.80 | 0.546 | 133/162 | 133/162 |
| bge_base_hybrid_d0.25_b0.75 | hybrid | 0.25 | 0.75 | 0.530 | 130/162 | 130/162 |
| bge_base_hybrid_d0.50_b0.50 | hybrid | 0.50 | 0.50 | 0.490 | 123/162 | 123/162 |
| bge_base_hybrid_d0.75_b0.25 | hybrid | 0.75 | 0.25 | 0.410 | 112/162 | 112/162 |

### Dev: Jina-v2-small (162 questions)

In [ ]:
if RUN_EXPENSIVE or (OUTPUT_DIR / "dev_summary.csv").is_file():
    display(model_table("jina"))
else:
    print("Dev results pending: Jina-v2-small")

Selected-metric preview of the saved results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| jina_dense | dense | N/A | N/A | 0.329 | 87/162 | 87/162 |
| jina_hybrid_d0.05_b0.95 | hybrid | 0.05 | 0.95 | 0.562 | 132/162 | 132/162 |
| jina_hybrid_d0.10_b0.90 | hybrid | 0.10 | 0.90 | 0.556 | 131/162 | 131/162 |
| jina_hybrid_d0.20_b0.80 | hybrid | 0.20 | 0.80 | 0.541 | 128/162 | 128/162 |
| jina_hybrid_d0.25_b0.75 | hybrid | 0.25 | 0.75 | 0.524 | 126/162 | 126/162 |
| jina_hybrid_d0.50_b0.50 | hybrid | 0.50 | 0.50 | 0.481 | 120/162 | 121/162 |
| jina_hybrid_d0.75_b0.25 | hybrid | 0.75 | 0.25 | 0.388 | 104/162 | 105/162 |

### Dev: GTE-ModernBERT (162 questions)

In [ ]:
if RUN_EXPENSIVE or (OUTPUT_DIR / "dev_summary.csv").is_file():
    display(model_table("gte_modernbert"))
else:
    print("Dev results pending: GTE-ModernBERT")

Selected-metric preview of the saved results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| gte_modernbert_dense | dense | N/A | N/A | 0.342 | 91/162 | 91/162 |
| gte_modernbert_hybrid_d0.05_b0.95 | hybrid | 0.05 | 0.95 | 0.554 | 135/162 | 135/162 |
| gte_modernbert_hybrid_d0.10_b0.90 | hybrid | 0.10 | 0.90 | 0.535 | 133/162 | 133/162 |
| gte_modernbert_hybrid_d0.20_b0.80 | hybrid | 0.20 | 0.80 | 0.529 | 128/162 | 128/162 |
| gte_modernbert_hybrid_d0.25_b0.75 | hybrid | 0.25 | 0.75 | 0.521 | 127/162 | 127/162 |
| gte_modernbert_hybrid_d0.50_b0.50 | hybrid | 0.50 | 0.50 | 0.479 | 122/162 | 123/162 |
| gte_modernbert_hybrid_d0.75_b0.25 | hybrid | 0.75 | 0.25 | 0.403 | 107/162 | 108/162 |

### Dev: Nomic-v1.5 (162 questions)

In [ ]:
if RUN_EXPENSIVE or (OUTPUT_DIR / "dev_summary.csv").is_file():
    display(model_table("nomic"))
else:
    print("Dev results pending: Nomic-v1.5")

Selected-metric preview of the saved results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| nomic_dense | dense | N/A | N/A | 0.334 | 96/162 | 96/162 |
| nomic_hybrid_d0.05_b0.95 | hybrid | 0.05 | 0.95 | 0.569 | 134/162 | 134/162 |
| nomic_hybrid_d0.10_b0.90 | hybrid | 0.10 | 0.90 | 0.562 | 132/162 | 132/162 |
| nomic_hybrid_d0.20_b0.80 | hybrid | 0.20 | 0.80 | 0.531 | 129/162 | 129/162 |
| nomic_hybrid_d0.25_b0.75 | hybrid | 0.25 | 0.75 | 0.520 | 127/162 | 127/162 |
| nomic_hybrid_d0.50_b0.50 | hybrid | 0.50 | 0.50 | 0.483 | 119/162 | 120/162 |
| nomic_hybrid_d0.75_b0.25 | hybrid | 0.75 | 0.25 | 0.413 | 112/162 | 113/162 |

### Dev: BGE-M3 (162 questions)

In [ ]:
if RUN_EXPENSIVE or (OUTPUT_DIR / "dev_summary.csv").is_file():
    display(model_table("bge_m3"))
else:
    print("Dev results pending: BGE-M3")

Selected-metric preview of the saved results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| bge_m3_dense | dense | N/A | N/A | 0.394 | 109/162 | 110/162 |
| bge_m3_hybrid_d0.05_b0.95 | hybrid | 0.05 | 0.95 | 0.564 | 133/162 | 133/162 |
| bge_m3_hybrid_d0.10_b0.90 | hybrid | 0.10 | 0.90 | 0.564 | 135/162 | 135/162 |
| bge_m3_hybrid_d0.20_b0.80 | hybrid | 0.20 | 0.80 | 0.558 | 132/162 | 133/162 |
| bge_m3_hybrid_d0.25_b0.75 | hybrid | 0.25 | 0.75 | 0.547 | 129/162 | 130/162 |
| bge_m3_hybrid_d0.50_b0.50 | hybrid | 0.50 | 0.50 | 0.492 | 122/162 | 123/162 |
| bge_m3_hybrid_d0.75_b0.25 | hybrid | 0.75 | 0.25 | 0.424 | 119/162 | 120/162 |

### Dev: Qwen3-Embedding-0.6B (162 questions)

In [ ]:
if RUN_EXPENSIVE or (OUTPUT_DIR / "dev_summary.csv").is_file():
    display(model_table("qwen3"))
else:
    print("Dev results pending: Qwen3-Embedding-0.6B")

Selected-metric preview of the saved results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| qwen3_dense | dense | N/A | N/A | 0.341 | 85/162 | 85/162 |
| qwen3_hybrid_d0.05_b0.95 | hybrid | 0.05 | 0.95 | 0.557 | 135/162 | 135/162 |
| qwen3_hybrid_d0.10_b0.90 | hybrid | 0.10 | 0.90 | 0.545 | 130/162 | 130/162 |
| qwen3_hybrid_d0.20_b0.80 | hybrid | 0.20 | 0.80 | 0.530 | 125/162 | 125/162 |
| qwen3_hybrid_d0.25_b0.75 | hybrid | 0.25 | 0.75 | 0.529 | 124/162 | 124/162 |
| qwen3_hybrid_d0.50_b0.50 | hybrid | 0.50 | 0.50 | 0.493 | 117/162 | 117/162 |
| qwen3_hybrid_d0.75_b0.25 | hybrid | 0.75 | 0.25 | 0.430 | 112/162 | 112/162 |

### Dev overall summary - all 64 configurations

All models and weights are in one table, ordered by **complete evidence@5**, then **MRR**, then run name. This is the same Dev selection order used for the frozen configuration. Every metric column remains available by scrolling horizontally.

In [ ]:
def overall_table(split: str) -> pd.DataFrame:
    assert split in ("dev", "test_all"), split
    live_name = "dev_summary" if split == "dev" else "test_all_summary"
    if live_name in globals():
        frame = globals()[live_name]
    else:
        saved = OUTPUT_DIR / f"{split}_summary.csv"
        if not saved.is_file():
            raise FileNotFoundError(f"Run the full comparison first: {saved}")
        frame = pd.read_csv(saved)
    table = frame.copy()
    table = table.sort_values(["complete_hit_at_5", "mrr", "run_name"],
                              ascending=[False, False, True]).reset_index(drop=True)
    table["mode"] = table["mode"].map(lambda mode: "hybrid" if mode.startswith("hybrid_") else mode)
    leading = ["run_name", "model", "mode", "dense_weight", "bm25_weight", "question_count"]
    columns = leading + [column for column in table.columns if column not in leading]
    return table[columns]

if "dev_summary" in globals() or (OUTPUT_DIR / "dev_summary.csv").is_file():
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(overall_table("dev"))
else:
    print("Dev overall summary pending. Run the full notebook first.")

## 9. Freeze a diagnostic configuration, then optional Test

First evaluate the single development-selected configuration, before inspecting any other Test results. The all-model diagnostic comparison follows in Section 10. Neither is a formal Test result because the Gold split and annotations are not frozen.

In [ ]:
if RUN_EXPENSIVE:
    # BM25 is a legitimate configuration: do not exclude a stronger baseline.
    winner = dev_summary.iloc[0].to_dict()
    frozen = {"model": winner["model"], "mode": winner["mode"],
              "bm25_weight": None if pd.isna(winner["bm25_weight"]) else float(winner["bm25_weight"]),
              "dense_weight": None if pd.isna(winner["dense_weight"]) else float(winner["dense_weight"]),
              "selected_on": "working Dev only",
              "selection_metric": "complete_hit_at_5 then mrr"}
    print("Frozen diagnostic configuration:", frozen)
    holdout_bm25 = {q["question_id"]: bm25_search(q["question"], SEARCH_DEPTH) for q in test_gold}
    if frozen["model"] != "bm25":
        model, index, _ = load_or_build_dense(frozen["model"])
    holdout_rows, holdout_candidates = [], []
    for q in test_gold:
        if frozen["model"] == "bm25":
            hits = holdout_bm25[q["question_id"]][:RETRIEVAL_DEPTH]
        else:
            dense = dense_search(q["question"], frozen["model"], model, index)
            hits = dense[:RETRIEVAL_DEPTH] if frozen["mode"] == "dense" else weighted_rrf(
                holdout_bm25[q["question_id"]], dense, frozen["bm25_weight"], frozen["dense_weight"])
        holdout_rows.append(score_one(q, hits, frozen["model"], frozen["mode"],
                                      frozen["bm25_weight"], frozen["dense_weight"]))
        for rank, (chunk_id, score) in enumerate(hits, 1):
            holdout_candidates.append({"question_id": q["question_id"], "model": frozen["model"],
                "mode": frozen["mode"], "rank": rank, "chunk_id": chunk_id, "score": score,
                "query": q["question"], "corpus_hash": corpus_manifest["corpus_hash"],
                "textbook_id": by_chunk[chunk_id]["textbook_id"], "provider": by_chunk[chunk_id]["provider"],
                "source_locator": by_chunk[chunk_id]["source_locator"]})
    if frozen["model"] != "bm25":
        del model, index
    display(summarize(holdout_rows))
else:
    frozen = None
    holdout_rows = holdout_candidates = []

## 10. Test (26 questions): all-model diagnostic comparison

This section runs BM25 and **all nine embedding models** on the 26 `proposed_test` questions. Each model has pure Dense plus the same six Dense/BM25 hybrids used for Dev. It is a diagnostic comparison only: these 26 questions are the user-designated working Test set, not the formal six-textbook Test set, and inspecting all configurations here must not be used to retune the Dev-selected configuration. The single configuration frozen before this comparison remains in Section 9.

In [ ]:
if RUN_EXPENSIVE:
    test_all_rows, test_all_candidates, test_index_provenance = evaluate_questions(
        test_gold, list(MODEL_SPECS), checkpoint_prefix="test_all")
    assert set(test_index_provenance) == set(index_provenance)
    test_all_summary = summarize(test_all_rows)
    print("Diagnostic Test configurations:", len(test_all_summary))
    print("Questions per configuration:", sorted(test_all_summary["question_count"].unique()))
else:
    test_all_rows = test_all_candidates = []
    print("No new Test comparison run; saved tables below are shown if available.")

### Test: BM25 baseline (26 questions; diagnostic)

In [ ]:
if "test_all_summary" in globals() or (OUTPUT_DIR / "test_all_summary.csv").is_file():
    display(model_table("bm25", split="test_all"))
else:
    print("Test comparison pending. Run the full notebook first.")

Selected-metric preview of the saved diagnostic Test results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| bm25_bm25 | bm25 | N/A | N/A | 0.480 | 17/26 | 18/26 |

### Test: MiniLM (26 questions; diagnostic)

In [ ]:
if "test_all_summary" in globals() or (OUTPUT_DIR / "test_all_summary.csv").is_file():
    display(model_table("minilm", split="test_all"))
else:
    print("Test comparison pending. Run the full notebook first.")

Selected-metric preview of the saved diagnostic Test results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| minilm_dense | dense | N/A | N/A | 0.125 | 6/26 | 6/26 |
| minilm_hybrid_d0.05_b0.95 | hybrid | 0.05 | 0.95 | 0.441 | 17/26 | 18/26 |
| minilm_hybrid_d0.10_b0.90 | hybrid | 0.10 | 0.90 | 0.414 | 17/26 | 18/26 |
| minilm_hybrid_d0.20_b0.80 | hybrid | 0.20 | 0.80 | 0.429 | 15/26 | 16/26 |
| minilm_hybrid_d0.25_b0.75 | hybrid | 0.25 | 0.75 | 0.391 | 13/26 | 14/26 |
| minilm_hybrid_d0.50_b0.50 | hybrid | 0.50 | 0.50 | 0.281 | 11/26 | 11/26 |
| minilm_hybrid_d0.75_b0.25 | hybrid | 0.75 | 0.25 | 0.171 | 9/26 | 9/26 |

### Test: MPNet (26 questions; diagnostic)

In [ ]:
if "test_all_summary" in globals() or (OUTPUT_DIR / "test_all_summary.csv").is_file():
    display(model_table("mpnet", split="test_all"))
else:
    print("Test comparison pending. Run the full notebook first.")

Selected-metric preview of the saved diagnostic Test results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| mpnet_dense | dense | N/A | N/A | 0.158 | 9/26 | 9/26 |
| mpnet_hybrid_d0.05_b0.95 | hybrid | 0.05 | 0.95 | 0.440 | 17/26 | 17/26 |
| mpnet_hybrid_d0.10_b0.90 | hybrid | 0.10 | 0.90 | 0.395 | 15/26 | 15/26 |
| mpnet_hybrid_d0.20_b0.80 | hybrid | 0.20 | 0.80 | 0.371 | 15/26 | 15/26 |
| mpnet_hybrid_d0.25_b0.75 | hybrid | 0.25 | 0.75 | 0.362 | 14/26 | 14/26 |
| mpnet_hybrid_d0.50_b0.50 | hybrid | 0.50 | 0.50 | 0.310 | 11/26 | 11/26 |
| mpnet_hybrid_d0.75_b0.25 | hybrid | 0.75 | 0.25 | 0.211 | 10/26 | 10/26 |

### Test: E5-base-v2 (26 questions; diagnostic)

In [ ]:
if "test_all_summary" in globals() or (OUTPUT_DIR / "test_all_summary.csv").is_file():
    display(model_table("e5_base", split="test_all"))
else:
    print("Test comparison pending. Run the full notebook first.")

Selected-metric preview of the saved diagnostic Test results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| e5_base_dense | dense | N/A | N/A | 0.229 | 11/26 | 11/26 |
| e5_base_hybrid_d0.05_b0.95 | hybrid | 0.05 | 0.95 | 0.485 | 17/26 | 18/26 |
| e5_base_hybrid_d0.10_b0.90 | hybrid | 0.10 | 0.90 | 0.488 | 17/26 | 18/26 |
| e5_base_hybrid_d0.20_b0.80 | hybrid | 0.20 | 0.80 | 0.485 | 17/26 | 18/26 |
| e5_base_hybrid_d0.25_b0.75 | hybrid | 0.25 | 0.75 | 0.485 | 17/26 | 18/26 |
| e5_base_hybrid_d0.50_b0.50 | hybrid | 0.50 | 0.50 | 0.406 | 17/26 | 18/26 |
| e5_base_hybrid_d0.75_b0.25 | hybrid | 0.75 | 0.25 | 0.267 | 15/26 | 15/26 |

### Test: BGE-base-en-v1.5 (26 questions; diagnostic)

In [ ]:
if "test_all_summary" in globals() or (OUTPUT_DIR / "test_all_summary.csv").is_file():
    display(model_table("bge_base", split="test_all"))
else:
    print("Test comparison pending. Run the full notebook first.")

Selected-metric preview of the saved diagnostic Test results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| bge_base_dense | dense | N/A | N/A | 0.243 | 11/26 | 11/26 |
| bge_base_hybrid_d0.05_b0.95 | hybrid | 0.05 | 0.95 | 0.468 | 17/26 | 18/26 |
| bge_base_hybrid_d0.10_b0.90 | hybrid | 0.10 | 0.90 | 0.436 | 16/26 | 16/26 |
| bge_base_hybrid_d0.20_b0.80 | hybrid | 0.20 | 0.80 | 0.402 | 16/26 | 16/26 |
| bge_base_hybrid_d0.25_b0.75 | hybrid | 0.25 | 0.75 | 0.383 | 16/26 | 16/26 |
| bge_base_hybrid_d0.50_b0.50 | hybrid | 0.50 | 0.50 | 0.299 | 12/26 | 12/26 |
| bge_base_hybrid_d0.75_b0.25 | hybrid | 0.75 | 0.25 | 0.276 | 11/26 | 11/26 |

### Test: Jina-v2-small (26 questions; diagnostic)

In [ ]:
if "test_all_summary" in globals() or (OUTPUT_DIR / "test_all_summary.csv").is_file():
    display(model_table("jina", split="test_all"))
else:
    print("Test comparison pending. Run the full notebook first.")

Selected-metric preview of the saved diagnostic Test results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| jina_dense | dense | N/A | N/A | 0.233 | 9/26 | 9/26 |
| jina_hybrid_d0.05_b0.95 | hybrid | 0.05 | 0.95 | 0.448 | 17/26 | 18/26 |
| jina_hybrid_d0.10_b0.90 | hybrid | 0.10 | 0.90 | 0.418 | 17/26 | 17/26 |
| jina_hybrid_d0.20_b0.80 | hybrid | 0.20 | 0.80 | 0.400 | 16/26 | 16/26 |
| jina_hybrid_d0.25_b0.75 | hybrid | 0.25 | 0.75 | 0.386 | 15/26 | 15/26 |
| jina_hybrid_d0.50_b0.50 | hybrid | 0.50 | 0.50 | 0.299 | 13/26 | 13/26 |
| jina_hybrid_d0.75_b0.25 | hybrid | 0.75 | 0.25 | 0.277 | 12/26 | 12/26 |

### Test: GTE-ModernBERT (26 questions; diagnostic)

In [ ]:
if "test_all_summary" in globals() or (OUTPUT_DIR / "test_all_summary.csv").is_file():
    display(model_table("gte_modernbert", split="test_all"))
else:
    print("Test comparison pending. Run the full notebook first.")

Selected-metric preview of the saved diagnostic Test results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| gte_modernbert_dense | dense | N/A | N/A | 0.356 | 13/26 | 14/26 |
| gte_modernbert_hybrid_d0.05_b0.95 | hybrid | 0.05 | 0.95 | 0.474 | 17/26 | 18/26 |
| gte_modernbert_hybrid_d0.10_b0.90 | hybrid | 0.10 | 0.90 | 0.459 | 17/26 | 17/26 |
| gte_modernbert_hybrid_d0.20_b0.80 | hybrid | 0.20 | 0.80 | 0.456 | 17/26 | 18/26 |
| gte_modernbert_hybrid_d0.25_b0.75 | hybrid | 0.25 | 0.75 | 0.450 | 17/26 | 18/26 |
| gte_modernbert_hybrid_d0.50_b0.50 | hybrid | 0.50 | 0.50 | 0.417 | 16/26 | 16/26 |
| gte_modernbert_hybrid_d0.75_b0.25 | hybrid | 0.75 | 0.25 | 0.387 | 15/26 | 16/26 |

### Test: Nomic-v1.5 (26 questions; diagnostic)

In [ ]:
if "test_all_summary" in globals() or (OUTPUT_DIR / "test_all_summary.csv").is_file():
    display(model_table("nomic", split="test_all"))
else:
    print("Test comparison pending. Run the full notebook first.")

Selected-metric preview of the saved diagnostic Test results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| nomic_dense | dense | N/A | N/A | 0.196 | 11/26 | 12/26 |
| nomic_hybrid_d0.05_b0.95 | hybrid | 0.05 | 0.95 | 0.474 | 17/26 | 18/26 |
| nomic_hybrid_d0.10_b0.90 | hybrid | 0.10 | 0.90 | 0.460 | 17/26 | 17/26 |
| nomic_hybrid_d0.20_b0.80 | hybrid | 0.20 | 0.80 | 0.455 | 16/26 | 16/26 |
| nomic_hybrid_d0.25_b0.75 | hybrid | 0.25 | 0.75 | 0.428 | 16/26 | 16/26 |
| nomic_hybrid_d0.50_b0.50 | hybrid | 0.50 | 0.50 | 0.373 | 15/26 | 15/26 |
| nomic_hybrid_d0.75_b0.25 | hybrid | 0.75 | 0.25 | 0.242 | 12/26 | 12/26 |

### Test: BGE-M3 (26 questions; diagnostic)

In [ ]:
if "test_all_summary" in globals() or (OUTPUT_DIR / "test_all_summary.csv").is_file():
    display(model_table("bge_m3", split="test_all"))
else:
    print("Test comparison pending. Run the full notebook first.")

Selected-metric preview of the saved diagnostic Test results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| bge_m3_dense | dense | N/A | N/A | 0.239 | 10/26 | 10/26 |
| bge_m3_hybrid_d0.05_b0.95 | hybrid | 0.05 | 0.95 | 0.469 | 17/26 | 17/26 |
| bge_m3_hybrid_d0.10_b0.90 | hybrid | 0.10 | 0.90 | 0.457 | 16/26 | 16/26 |
| bge_m3_hybrid_d0.20_b0.80 | hybrid | 0.20 | 0.80 | 0.430 | 16/26 | 16/26 |
| bge_m3_hybrid_d0.25_b0.75 | hybrid | 0.25 | 0.75 | 0.446 | 16/26 | 16/26 |
| bge_m3_hybrid_d0.50_b0.50 | hybrid | 0.50 | 0.50 | 0.347 | 13/26 | 13/26 |
| bge_m3_hybrid_d0.75_b0.25 | hybrid | 0.75 | 0.25 | 0.253 | 12/26 | 12/26 |

### Test: Qwen3-Embedding-0.6B (26 questions; diagnostic)

In [ ]:
if "test_all_summary" in globals() or (OUTPUT_DIR / "test_all_summary.csv").is_file():
    display(model_table("qwen3", split="test_all"))
else:
    print("Test comparison pending. Run the full notebook first.")

Selected-metric preview of the saved diagnostic Test results:

| Run | Mode | Dense | BM25 | MRR | Complete evidence@5 | Hit@5 |
|---|---|---:|---:|---:|---:|---:|
| qwen3_dense | dense | N/A | N/A | 0.272 | 11/26 | 11/26 |
| qwen3_hybrid_d0.05_b0.95 | hybrid | 0.05 | 0.95 | 0.500 | 17/26 | 17/26 |
| qwen3_hybrid_d0.10_b0.90 | hybrid | 0.10 | 0.90 | 0.479 | 18/26 | 18/26 |
| qwen3_hybrid_d0.20_b0.80 | hybrid | 0.20 | 0.80 | 0.459 | 18/26 | 18/26 |
| qwen3_hybrid_d0.25_b0.75 | hybrid | 0.25 | 0.75 | 0.423 | 17/26 | 17/26 |
| qwen3_hybrid_d0.50_b0.50 | hybrid | 0.50 | 0.50 | 0.371 | 15/26 | 15/26 |
| qwen3_hybrid_d0.75_b0.25 | hybrid | 0.75 | 0.25 | 0.310 | 14/26 | 14/26 |

### Test overall summary - all 64 configurations

All nine models, six hybrids per model, pure Dense and BM25 are in one table, ordered by **complete evidence@5**, then **MRR**, then run name. This order is for inspection only: the Test results were **not** used to change the Dev-frozen configuration. Every metric column remains available by scrolling horizontally.

In [ ]:
if "test_all_summary" in globals() or (OUTPUT_DIR / "test_all_summary.csv").is_file():
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(overall_table("test_all"))
else:
    print("Test overall summary pending. Run the full notebook first.")

## 11. Save auditable diagnostic outputs and manifest

In [ ]:
def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        "".join(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n" for row in rows),
        encoding="utf-8",
    )

def candidate_set_id(rows: list[dict]) -> str:
    payload = json.dumps(rows, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return sha256(payload)

if RUN_EXPENSIVE:
    expected_configurations = 1 + len(MODEL_SPECS) * (1 + len(HYBRID_WEIGHTS))
    assert len(dev_rows) == len(dev_gold) * expected_configurations
    assert len(test_all_rows) == len(test_gold) * expected_configurations
    assert {row["question_id"] for row in dev_rows} == {row["question_id"] for row in dev_gold}
    assert {row["question_id"] for row in holdout_rows} == {row["question_id"] for row in test_gold}
    assert {row["question_id"] for row in test_all_rows} == {row["question_id"] for row in test_gold}
    assert all(row["evaluation_split"] == "test" for row in test_all_rows)

    frozen_test_rows = {
        row["question_id"]: row for row in test_all_rows
        if row["model"] == frozen["model"] and row["mode"] == frozen["mode"]
    }
    assert len(frozen_test_rows) == len(test_gold)
    assert all(
        row["ranked_chunk_ids"] == frozen_test_rows[row["question_id"]]["ranked_chunk_ids"]
        for row in holdout_rows
    ), "Frozen result must agree with the all-model Test audit."

    selected_dev_candidates = [
        row for row in dev_candidates
        if row["model"] == frozen["model"] and row["mode"] == frozen["mode"]
    ]
    selected_test_candidates = [
        row for row in test_all_candidates
        if row["model"] == frozen["model"] and row["mode"] == frozen["mode"]
    ]
    dev_candidate_set_id = candidate_set_id(selected_dev_candidates)
    test_candidate_set_id = candidate_set_id(selected_test_candidates)
    for row in selected_dev_candidates:
        row.update({"condition_id": "baseline_initial_candidates", "candidate_set_id": dev_candidate_set_id})
    for row in selected_test_candidates:
        row.update({"condition_id": "baseline_initial_candidates", "candidate_set_id": test_candidate_set_id})

    retrieval_misses = [
        {
            "question_id": row["question_id"],
            "evaluation_split": row["evaluation_split"],
            "model": row["model"],
            "mode": row["mode"],
            "hit_at_5": row["hit_at_5"],
            "complete_hit_at_5": row["complete_hit_at_5"],
            "first_relevant_rank": row["first_relevant_rank"],
            "complete_evidence_rank": row["complete_evidence_rank"],
        }
        for row in dev_rows
        if row["model"] == frozen["model"] and row["mode"] == frozen["mode"]
        and (row["hit_at_5"] == 0 or row["complete_hit_at_5"] == 0)
    ]

    output_rows = {
        "gold_span_bindings.jsonl": binding_rows,
        "dev_metrics.jsonl": dev_rows,
        "dev_candidate_hits.jsonl": dev_candidates,
        "baseline_initial_candidates_dev.jsonl": selected_dev_candidates,
        "test_frozen_metrics.jsonl": holdout_rows,
        "test_frozen_candidate_hits.jsonl": holdout_candidates,
        "test_all_metrics.jsonl": test_all_rows,
        "test_all_candidate_hits.jsonl": test_all_candidates,
        "baseline_initial_candidates_test.jsonl": selected_test_candidates,
        "retrieval_misses.jsonl": retrieval_misses,
        "failures.jsonl": [],
    }
    for name, rows in output_rows.items():
        write_jsonl(OUTPUT_DIR / name, rows)

    dev_summary_table = summarize(dev_rows)
    test_summary_table = summarize(test_all_rows)
    dev_summary_table.to_csv(OUTPUT_DIR / "dev_summary.csv", index=False)
    dev_summary_table.sort_values(
        ["complete_hit_at_5", "mrr", "run_name"], ascending=[False, False, True]
    ).to_csv(OUTPUT_DIR / "dev_overall_ranked.csv", index=False)
    grouped_summary(dev_rows).to_csv(OUTPUT_DIR / "dev_grouped_summary.csv", index=False)
    summarize(holdout_rows).to_csv(OUTPUT_DIR / "test_frozen_summary.csv", index=False)
    test_summary_table.to_csv(OUTPUT_DIR / "test_all_summary.csv", index=False)
    test_summary_table.sort_values(
        ["complete_hit_at_5", "mrr", "run_name"], ascending=[False, False, True]
    ).to_csv(OUTPUT_DIR / "test_all_overall_ranked.csv", index=False)

    output_file_sha256 = {
        name: sha256((OUTPUT_DIR / name).read_bytes()) for name in output_rows
    }
    run_manifest = {
        "schema_version": "m6-w6-v2-diagnostic-2",
        "reportable": False,
        "condition_id": "baseline_initial_candidates",
        "method_config": METHOD_CONFIG,
        "evaluation_set_version": EVALUATION_SET_VERSION,
        "gold_acceptance_policy": GOLD_ACCEPTANCE_POLICY,
        "source_annotation_status_counts": dict(collections.Counter(row["annotation_status"] for row in gold)),
        "gold_annotation_version": GOLD_ANNOTATION_VERSION,
        "gold_sha256": sha256(gold_bytes),
        "binding_sha256": sha256(binding_bytes),
        "github_commit": repo_commit,
        "corpus_version": corpus_manifest["corpus_version"],
        "corpus_hash": corpus_manifest["corpus_hash"],
        "corpus_manifest_hash": corpus_manifest["manifest_hash"],
        "chunk_config_hash": corpus_manifest["chunk_config_hash"],
        "required_textbook_ids": corpus_manifest["required_textbook_ids"],
        "included_textbook_ids": corpus_manifest["included_textbook_ids"],
        "corpus_zip_sha256": sha256(CORPUS_ZIP.read_bytes()),
        "m5_zip_sha256": sha256(M5_ZIP.read_bytes()),
        "indexes": {
            key: {
                "embedding_model": artifact.get("embedding_model", artifact.get("model")),
                "embedding_revision": artifact.get("embedding_revision"),
                "index_sha256": artifact.get("metadata", {}).get("index_sha256", artifact.get("index_sha256")),
                "normalise_embeddings": artifact.get("normalise_embeddings"),
                "similarity_metric": artifact.get("similarity_metric", artifact.get("metric")),
            }
            for key, artifact in index_provenance.items()
        },
        "frozen_diagnostic_configuration": frozen,
        "dev_candidate_set_id": dev_candidate_set_id,
        "test_candidate_set_id": test_candidate_set_id,
        "dev_question_count": len(dev_gold),
        "test_question_count": len(test_gold),
        "test_all_configuration_count": len(test_summary_table),
        "test_all_is_diagnostic": True,
        "failure_count": 0,
        "retrieval_miss_count": len(retrieval_misses),
        "unanswerable_question_count": 0,
        "unsupported_metrics": [
            "unanswerable_no_hit_rate", "unanswerable_false_positive_rate",
            "cross_textbook_rate", "cross_provider_rate",
        ],
        "limitations": [
            "user-designated working Dev/Test split", "development corpus", "three textbooks only",
            "all Gold rows are answerable", "source annotation status remains draft; user designated the dataset for M6 scoring",
        ],
        "output_file_sha256": output_file_sha256,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    (OUTPUT_DIR / "run_manifest.json").write_text(
        json.dumps(run_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    print("Saved M6 handoff:", OUTPUT_DIR)
    print("Dev candidate set:", dev_candidate_set_id)
    print("Test candidate set:", test_candidate_set_id)
else:
    print("Preflight complete; no measured files were written. Set CS30_M6_RUN_MODELS=1 for the full run.")


## Interpretation guardrails

**For actual model scores, use the per-model development tables above.** The notes below describe interpretation limits, not additional scores.

- This notebook compares retrieval only; it does not evaluate M7/M8 answer quality or refusal behavior.
- All 188 integrated questions are answerable. `unanswerable_no_hit_rate` and `unanswerable_false_positive_rate` are **not applicable**, not zero. Empty retrieval and false no-relevant-hit on answerable questions are measured separately.
- The Gold evidence in this set is single-source. Cross-textbook/cross-provider performance cannot be inferred from a group with no such questions. The grouped CSV exposes the actual denominators.
- `dev_candidate_hits.jsonl` and `test_frozen_candidate_hits.jsonl` are real ranked candidate handoffs with chunk ID, source and score, generated only after model execution.
- The integrated file is accepted as working Gold for this M6 experiment; its source `annotation_status` field is retained for traceability and is not a scoring gate.
- For a formal run, obtain an official Dev/Test split, all required sources including CK-12, rebuilt indexes for the exact corpus, and approved evaluation-set/Gold versions, then enable and satisfy `FORMAL_MODE`.